In [1]:
import torch
from pathlib import Path

from utils import train

In [2]:
BASE_DIR = Path.cwd().parent
print(BASE_DIR)

/home/evgeniy/Документы/GitHub/YandexNN/sprint_4


data: https://code.s3.yandex.net/deep-learning/archive_lesson4.zip

In [3]:
dataset_path = BASE_DIR / "data" / "multimodal"
images_path = dataset_path / "images"
df_path = dataset_path / "items.csv"

In [4]:
class Config:
    # для воспроизводимости
    SEED = 42

    # Модели
    TEXT_MODEL_NAME = "bert-base-uncased"
    IMAGE_MODEL_NAME = "tf_efficientnet_b0"

    # Какие слои размораживаем - совпадают с нэймингом в моделях
    TEXT_MODEL_UNFREEZE = "encoder.layer.11|pooler"
    IMAGE_MODEL_UNFREEZE = "blocks.6|conv_head|bn2"
    
    # Гиперпараметры
    BATCH_SIZE = 256 
    TEXT_LR = 3e-5
    IMAGE_LR = 1e-4
    CLASSIFIER_LR = 1e-3
    EPOCHS = 30
    DROPOUT = 0.3
    HIDDEN_DIM = 256
    NUM_CLASSES = 4

    # Пути
    TRAIN_DF_PATH = dataset_path / "imdb_train.csv"
    VAL_DF_PATH = dataset_path / "imdb_val.csv"
    SAVE_PATH = BASE_DIR / "models" / "multimodal" / "best_model.pth"

In [6]:
cfg = Config()

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


# Эксперимент 1 (без масок по тексту или изображениям)

In [8]:
train(cfg, device, mask=None)

![Без масок](../docs/exp1_multimodul.png)

# Эксперимент 2 (маска по изображениям)

In [ ]:
train(cfg, device, mask="image")

![Маска по изображению](../docs/exp2_multimodul.png)

# Эксперимент 3 (маска по тексту)

In [11]:
train(cfg, device, mask="text")

![Маска по тексту](../docs/exp3_multimodul.png)

# Сравнение

**Сходимость и стабильность**
Все три запуска демонстрируют идентичную динамику обучения: пик валидационной метрики достигается на 27-й эпохе (Val F1 = 0.7900–0.7950), после чего наблюдается стабилизация или незначительный спад. Кривые потерь монотонно убывают от 1.42 до ~0.39–0.40, что свидетельствует о воспроизводимости процесса обучения.

**Качество моделей**
Запуск 2 превосходит остальные по ключевым показателям: лучшая валидационная метрика (0.7950 против 0.7900) и наименьший разрыв между тренировочной и валидационной выборками (0.074 против 0.081–0.082). Запуски 1 и 3 практически идентичны по всем метрикам (различия <0.001), что подтверждает детерминированность обучения при фиксированных гиперпараметрах.

**Переобучение**
Во всех случаях признаки переобучения проявляются после 20-й эпохи: тренировочная метрика продолжает рост (до 0.86–0.87), тогда как валидационная выходит на плато. Запуск 2 демонстрирует наилучшую обобщающую способность — наименьшую дивергенцию между Train и Val F1.